# Figure creation with new atlases (human HCP-MMP1 + macaque MacBNA)

This notebook is the runnable companion to the
[Figure creation](../docs/tutorials/figure_creation.md) tutorial. It extracts
**real** ROI coordinates/names from two atlas volumes, fabricates **synthetic**
networks (fixed random seeds), and renders every figure used on the docs page.

> All connectivity matrices, modules, node sizes and metrics here are synthetic —
> only the coordinates and ROI names are real. The atlas volumes/meshes are large
> external data you supply under `test_files/tutorial_files/parcellation and meshes/`.

In [ ]:
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image as IPyImage

# Walk up to the repo root so relative paths resolve from anywhere.
here = Path.cwd()
repo_root = here
for p in [here, *here.parents]:
    if (p / 'test_files').is_dir() and (p / 'tutorial').is_dir():
        repo_root = p
        break
print('Repo root:', repo_root)

TF = repo_root / 'test_files' / 'tutorial_files'
DEMO = TF / 'new_atlas_demo'
PARC = TF / 'parcellation and meshes'
IMG = repo_root / 'docs' / 'images' / 'figure_creation'
IMG.mkdir(parents=True, exist_ok=True)
HTML_TMP = Path(tempfile.mkdtemp(prefix='figcreate_'))   # throwaway HTML dummies
sys.path.append(str(DEMO))

from HarrisLabPlotting import (
    load_mesh_file,
    create_brain_connectivity_plot,
    create_brain_connectivity_plot_with_modularity,
)

HUMAN_MESH = PARC / 'HCPMMP1_on_MNI152_ICBM2009a_nlin_hd_0.obj'
MONKEY_MESH = PARC / 'monkey_brain_mesh_MacBNA.obj'
EXAMPLE_EDGE = TF / 'node_edge_28' / 'connectivity_28.edge'
print('Inputs present:',
      HUMAN_MESH.exists(), MONKEY_MESH.exists(), EXAMPLE_EDGE.exists())

## 0. Generate the data

`generate_figure_data.py` builds both LUTs (from each atlas's own label table),
extracts real ROI coordinates, and fabricates the synthetic networks — all
deterministically. It also runs an alignment sanity-check at the end.

In [ ]:
import generate_figure_data as gfd
gfd.main()

## 1. Human — modularity across six views (2x3 grid)

A clean 2x3 multi-view grid of a 50-edge / 5-module synthetic network: nodes
colored by module (the default), no title and no edge-width key, module legend in
the first panel only.

In [ ]:
hv, hf = load_mesh_file(str(HUMAN_MESH))
coords = pd.read_csv(DEMO / 'human' / 'hcpmmp1_coords.csv')

out = IMG / 'human_modularity_grid_2x3.png'
create_brain_connectivity_plot_with_modularity(
    vertices=hv, faces=hf, roi_coords_df=coords,
    connectivity_matrix=str(DEMO / 'human' / 'hcpmmp1_modular_network.csv'),
    module_assignments=str(DEMO / 'human' / 'hcpmmp1_modules.csv'),
    multi_view=['anterior', 'posterior', 'left', 'right', 'superior', 'oblique'],
    multi_view_grid=(2, 3), multi_view_panel_size=(700, 700),
    show_node_labels=True, label_font_size=9, show_width_legend=False, plot_title='',
    zoom=1.3, image_dpi=300,
    save_path=str(HTML_TMP / 'human_grid.html'), export_image=str(out),
    export_show_legend=False, mesh_opacity= 0.1
)
IPyImage(filename=str(out))

## 2. Monkey — per-node sizes and legend keys

Reuse the bundled 28-node example topology on 28 real MacBNA ROIs. Node sizes are
pre-scaled from a synthetic participation coefficient.

In [ ]:
mv, mf = load_mesh_file(str(MONKEY_MESH))
mcoords = pd.read_csv(DEMO / 'monkey' / 'coords_28.csv')
sizes = str(DEMO / 'monkey' / 'sizes_from_pc.csv')
metrics = str(DEMO / 'monkey' / 'metrics.csv')

common = dict(
    vertices=mv, faces=mf, roi_coords_df=mcoords,
    connectivity_matrix=str(EXAMPLE_EDGE), camera_view='left',
    show_node_labels=False, image_dpi=150, zoom=1.2,
)

# (a) vector sizes + scaled edges -> both keys appear automatically
out = IMG / 'monkey_size_key.png'
create_brain_connectivity_plot(
    node_size=sizes, edge_width=(1.0, 8.0), node_size_scale=0.5,
    save_path=str(HTML_TMP / 'm.html'), export_image=str(out), **common)
IPyImage(filename=str(out))

In [ ]:
# (b) scalar size + fixed width -> both keys auto-skipped
out = IMG / 'monkey_no_keys.png'
create_brain_connectivity_plot(
    node_size=10, edge_width=2.0,
    save_path=str(HTML_TMP / 'm.html'), export_image=str(out), **common)
IPyImage(filename=str(out))

In [ ]:
# (c) metric-labeled size key (participation coefficient)
out = IMG / 'monkey_metric_key.png'
create_brain_connectivity_plot(
    node_size=sizes, node_metrics=metrics,
    node_size_legend_metric='participation_coef',
    edge_width=(1.0, 8.0), node_size_scale=0.5,
    save_path=str(HTML_TMP / 'm.html'), export_image=str(out), **common)
IPyImage(filename=str(out))

## 3. Monkey — default vs. customized

The same network rendered two ways: default (scalar size, fixed width, purple
nodes) vs. customized (PC-scaled sizes, module colors + border, scaled edges,
metric-labeled key).

In [ ]:
modules = (np.arange(len(mcoords)) % 4) + 1  # synthetic 4-module coloring

create_brain_connectivity_plot(
    node_size=8, edge_width=2.0,
    save_path=str(HTML_TMP / 'm.html'), export_image=str(IMG / 'monkey_default.png'), **common)

create_brain_connectivity_plot(
    node_size=sizes, node_size_scale=0.5,
    node_color=modules, node_border_color='black',
    node_metrics=metrics, node_size_legend_metric='participation_coef',
    edge_width=(1.0, 8.0),
    save_path=str(HTML_TMP / 'm.html'), export_image=str(IMG / 'monkey_customized.png'), **common)

IPyImage(filename=str(IMG / 'monkey_customized.png'))

---

**Tip:** before plotting any new atlas, run the pre-flight checks in
[Checking atlas/mesh alignment](../docs/how_to/check_atlas_mesh_alignment.md), e.g.

```bash
hlplot utils check-alignment \
  --coords test_files/tutorial_files/new_atlas_demo/human/hcpmmp1_coords.csv \
  --mesh "test_files/tutorial_files/parcellation and meshes/HCPMMP1_on_MNI152_ICBM2009a_nlin_hd_0.obj"
```

## 4. Modularity visualization types (114-ROI k5 example)

The `viz_type` / `inter_edge_color` / `node_roles` knobs render the **same** *k=5*
community-detection result (bundled `brain_mesh.gii` + `k5_state_0/`, 114 ROIs,
6 modules) several different ways. Each is exported as a 3-view multi-view PNG, a
single superior view, and an interactive HTML (HTMLs go to a temp dir).

In [ ]:
K5 = TF / 'k5_state_0'
K5_COORDS = TF / 'output' / 'atlas_114_test' / 'atlas_114_test_comma.csv'
K5_IMG = IMG / 'k5'
K5_IMG.mkdir(parents=True, exist_ok=True)

bv, bf = load_mesh_file(str(TF / 'brain_mesh.gii'))
k5coords = pd.read_csv(K5_COORDS)
k5base = dict(
    vertices=bv, faces=bf, roi_coords_df=k5coords,
    connectivity_matrix=str(K5 / 'connectivity_matrix.csv'),
    module_assignments=str(K5 / 'module_assignments.csv'),
    node_metrics=str(K5 / 'combined_metrics.csv'),
    node_size=10, image_dpi=150, show_node_labels=False,
)
k5types = [
    ('default', 'all edges', dict(viz_type='all')),
    ('all_inter_black', 'all edges, inter black',
     dict(viz_type='all', edge_color_mode='module', inter_edge_color='black')),
    ('intra', 'intra-module edges', dict(viz_type='intra')),
    ('inter', 'inter-module edges', dict(viz_type='inter')),
    ('inter_black', 'inter-module edges (black)',
     dict(viz_type='inter', edge_color_mode='module', inter_edge_color='black')),
    ('nodes_only', 'nodes only', dict(viz_type='nodes_only', show_width_legend=False)),
    ('nodal_roles', 'nodal roles',
     dict(viz_type='nodes_only', node_roles=True, show_width_legend=False)),
]
for key, title, kw in k5types:
    create_brain_connectivity_plot_with_modularity(
        **k5base, **kw, plot_title=f'Modularity \u2014 {title}',
        multi_view=['left', 'superior', 'posterior'], multi_view_panel_size=(700, 700),
        save_path=str(HTML_TMP / f'k5_{key}.html'),
        export_image=str(K5_IMG / f'{key}_multiview.png'))
    create_brain_connectivity_plot_with_modularity(
        **k5base, **kw, plot_title=f'Modularity \u2014 {title}', camera_view='superior',
        save_path=str(HTML_TMP / f'k5_{key}_sup.html'),
        export_image=str(K5_IMG / f'{key}_superior.png'))
print('k5 viz-type figures ->', K5_IMG)
IPyImage(filename=str(K5_IMG / 'nodal_roles_superior.png'))

---

## Tuning controls

The sections below (species grid, p-value scaling, nodal roles) each open with a
**PARAMETERS** block you can edit, then render. Flip `DRAFT` for fast 150-DPI
iteration while you dial things in; set it to `False` for the 600-DPI publication
renders. Single-image PNG exports are auto-cropped on a square canvas
(`export_autocrop=True`, the default) so they stay tight and do **not** squish
when you raise the DPI.

In [ ]:
DRAFT = False                         # True = fast 150-DPI drafts; False = 600-DPI publication
IMAGE_DPI = 150 if DRAFT else 600
print(f"DRAFT={DRAFT} -> IMAGE_DPI={IMAGE_DPI}")

## 5. Cross-species comparison grid (human / rat / macaque)

A 2x3 grid whose **columns are three different meshes** (human / rat / macaque) and
whose six cells are the six canonical BrainNet views (row 1 = left / superior /
right, row 2 = anterior / inferior / posterior). Each species shows a minimal
module-colored network on its own mesh. Each panel is rendered separately, then
composed with `compose_image_grid` (the engine behind `hlplot montage`). Three versions are produced: ROI labels off, full roi_name labels, and
short-form labels (roi_name minus the hemisphere suffix, e.g. `V1_L`->`V1`).

Tweak `SPECIES_ZOOM` if a labeled panel looks too zoomed in / labels clip.



In [ ]:
# ================= PARAMETERS (edit me) =================
# Per-(species, view) camera zoom. A view whose brain fills more of the panel
# needs a smaller zoom or it reads as "too zoomed in" next to the others.
SPECIES_ZOOM = {
    ('Human', 'left'): 1.0,     ('Human', 'anterior'): 1.0,
    ('Rat', 'superior'): 1.2,   ('Rat', 'inferior'): 1.0,
    ('Macaque', 'right'): 1.2,  ('Macaque', 'posterior'): 1.2,
}
SPECIES_PANEL  = (500, 500)   # per-panel px (before DPI supersample)
SPECIES_NODE   = 10           # node marker size
SPECIES_EDGE_W = 2.0          # fixed edge width (minimal look)
LABEL_FONT     = 9            # ROI-label font size
# Versions to render: (tag, label mode) -- 'none' | 'full' | 'short'
VERSIONS = [('nolabels', 'none'), ('labeled', 'full'), ('shortform', 'short')]
# =======================================================
import re
from HarrisLabPlotting import compose_image_grid

SP_IMG = IMG / 'species'; SP_IMG.mkdir(parents=True, exist_ok=True)
_HEMI_RE = re.compile(r'_(?:L|R|left|right)$', re.IGNORECASE)

def short_name(name):
    """V1_L -> V1 ; AUD_left -> AUD ; IFG.cv_left -> IFG.cv"""
    return _HEMI_RE.sub('', str(name))

SPECIES = [
    dict(name='Human',   mesh=HUMAN_MESH,          coords=DEMO/'human'/'hcpmmp1_coords.csv',
         matrix=DEMO/'human'/'hcpmmp1_modular_network.csv',
         modules=DEMO/'human'/'hcpmmp1_modules.csv', views=['left', 'anterior']),
    dict(name='Rat',     mesh=TF/'brain_mesh.gii',  coords=TF/'output'/'atlas_28_test_comma.csv',
         matrix=EXAMPLE_EDGE, modules=TF/'node_edge_28'/'modules_28.csv',
         views=['superior', 'inferior']),
    dict(name='Macaque', mesh=MONKEY_MESH,          coords=DEMO/'monkey'/'coords_28.csv',
         matrix=EXAMPLE_EDGE, modules=None, views=['right', 'posterior']),  # modules synthesized
]
sp_meshes = {s['name']: load_mesh_file(str(s['mesh'])) for s in SPECIES}

def render_species_grid(tag, label_mode):
    cells = {}
    for si, s in enumerate(SPECIES):
        c = pd.read_csv(s['coords'])
        if label_mode == 'short':
            c = c.copy(); c['roi_name'] = c['roi_name'].map(short_name)
        mods = str(s['modules']) if s['modules'] is not None else (np.arange(len(c)) % 4) + 1
        vv, ff = sp_meshes[s['name']]
        for row, view in enumerate(s['views']):
            out = HTML_TMP / f"sp_{s['name']}_{view}_{label_mode}.png"
            create_brain_connectivity_plot_with_modularity(
                vertices=vv, faces=ff, roi_coords_df=c,
                connectivity_matrix=str(s['matrix']), module_assignments=mods,
                node_size=SPECIES_NODE, edge_width=SPECIES_EDGE_W,
                show_node_labels=(label_mode != 'none'), label_font_size=LABEL_FONT,
                show_width_legend=False, plot_title='',
                multi_view=[view], multi_view_panel_size=SPECIES_PANEL,
                multi_view_keep_first_legend=False, multi_view_panel_labels=[''],
                image_dpi=IMAGE_DPI, zoom=SPECIES_ZOOM[(s['name'], view)],
                save_path=str(HTML_TMP / 'sp.html'), export_image=str(out))
            cells[(row, si)] = out
    imgs = [cells[(r, si)] for r in (0, 1) for si in range(len(SPECIES))]
    return compose_image_grid(
        imgs, SP_IMG / f'species_grid_{tag}.png', grid=(2, 3),
        col_labels=['Human', 'Rat', 'Macaque'],
        panel_labels=['Left', 'Superior', 'Right', 'Anterior', 'Inferior', 'Posterior'])

for _tag, _mode in VERSIONS:
    render_species_grid(_tag, _mode)
    print('wrote', f'species_grid_{_tag}.png')
IPyImage(filename=str(SP_IMG / 'species_grid_shortform.png'))

## 6. Scaling edges and nodes by p-value significance

A p-value network can encode significance twice: **edge width** by `-log10(p)`
(built in via `matrix_type='pvalue'`) and **node size** by a per-node significance
you derive. The pair below contrasts a flat baseline (uniform) with the fully
scaled figure. Both are single-image exports, so they benefit from the
square-canvas auto-crop (tight, dpi-stable).




In [ ]:
# ================= PARAMETERS (edit me) =================
PVALUE_THRESHOLD = 0.05          # edges with p > threshold are dropped
EDGE_W_MIN, EDGE_W_MAX = 1.0, 9.0    # scaled edge-width range (~ -log10 p)
SIZE_MIN, SIZE_MAX = 6.0, 24.0       # node px range (~ per-node significance)
NODE_SIZE_SCALE = 1.0            # extra multiplier on the derived node sizes
EDGE_WIDTH_SCALE = 2             # uniform multiplier on every edge width
PVAL_CAMERA = 'superior'
SHOW_NODE_LABELS = True          # show ROI names
SHORT_LABELS = True              # shorten the hemisphere suffix to cut overlap
KEEP_HEMISPHERE = True           # ...but keep it as _L/_R so left/right pairs stay distinct
MULTI_VIEW = ['left', 'superior', 'posterior']   # panels for the multi-view strip
MULTI_VIEW_PANEL = (700, 700)
LABEL_FONT_SIZE = 7              # small enough for 28 labels
# =======================================================
PV_IMG = IMG / 'pvalue'; PV_IMG.mkdir(parents=True, exist_ok=True)
rat_v, rat_f = load_mesh_file(str(TF / 'brain_mesh.gii'))
pcoords = pd.read_csv(TF / 'output' / 'atlas_28_test_comma.csv')
if SHORT_LABELS:
    from HarrisLabPlotting import short_roi_name
    pcoords = pcoords.copy()
    pcoords['roi_name'] = pcoords['roi_name'].map(
        lambda n: short_roi_name(n, keep_hemisphere=KEEP_HEMISPHERE))
PVALS = TF / 'node_edge_28' / 'pvalues_28_spread.csv'   # significance spread over ~5 orders of magnitude

# per-node significance = sum of -log10(p) over each node's surviving edges
P = np.loadtxt(PVALS, delimiter=',')
Wsig = np.where((P > 0) & (P <= PVALUE_THRESHOLD), -np.log10(np.clip(P, 1e-300, 1.0)), 0.0)
np.fill_diagonal(Wsig, 0.0)
sig = Wsig.sum(axis=1)
node_px = (SIZE_MIN + (sig - sig.min()) / (sig.max() - sig.min()) * (SIZE_MAX - SIZE_MIN)) * NODE_SIZE_SCALE

pv_common = dict(vertices=rat_v, faces=rat_f, roi_coords_df=pcoords,
                 connectivity_matrix=str(PVALS), matrix_type='pvalue',
                 pvalue_threshold=PVALUE_THRESHOLD,
                 edge_width_scale=EDGE_WIDTH_SCALE, image_dpi=IMAGE_DPI,
                 show_node_labels=SHOW_NODE_LABELS, label_font_size=LABEL_FONT_SIZE)

# The two variants: (a) nothing encodes significance, (b) edges AND nodes do.
variants = {
    'pval_uniform': dict(edge_width=2.0, node_size=8,      # fixed + scalar -> keys auto-skip
                         plot_title='p-values, uniform'),
    'pval_scaled': dict(edge_width=(EDGE_W_MIN, EDGE_W_MAX), node_size=node_px,
                        node_metrics=pd.DataFrame({'roi_name': pcoords['roi_name'],
                                                   'node_significance': sig}),
                        node_size_legend_metric='node_significance',
                        plot_title='p-values, significance-scaled'),
}
for _key, _kw in variants.items():
    # single view
    create_brain_connectivity_plot(**pv_common, **_kw, camera_view=PVAL_CAMERA,
        save_path=str(HTML_TMP / f'{_key}.html'),
        export_image=str(PV_IMG / f'{_key}.png'))
    # multi-view strip (keys stay on the first panel)
    create_brain_connectivity_plot(**pv_common, **_kw,
        multi_view=MULTI_VIEW, multi_view_panel_size=MULTI_VIEW_PANEL,
        save_path=str(HTML_TMP / f'{_key}_mv.html'),
        export_image=str(PV_IMG / f'{_key}_multiview.png'))
    print('wrote', f'{_key}.png', '+', f'{_key}_multiview.png')
IPyImage(filename=str(PV_IMG / 'pval_scaled_multiview.png'))

## 7. Nodal roles — with and without edges (k5)

`node_roles=True` classifies each node by the Guimera-Amaral cartographic two-cut
(needs `node_metrics` with `participation_coef` + `within_module_zscore`) and draws
the role as a colored **border ring**; the node fill stays its module color. It
composes with any `viz_type`, so you can show roles with **no edges**
(`viz_type='nodes_only'`) or with the **full edge set** (`viz_type='all'`).

In [ ]:
# ================= PARAMETERS (edit me) =================
K5_ROLE_VIEWS = ['left', 'superior', 'posterior']   # multi-view panels
K5_ROLE_PANEL = (700, 700)
K5_NODE = 8
# =======================================================
K5r = TF / 'k5_state_0'
K5r_coords = pd.read_csv(TF / 'output' / 'atlas_114_test' / 'atlas_114_test_comma.csv')
K5r_IMG = IMG / 'k5'; K5r_IMG.mkdir(parents=True, exist_ok=True)
bv2, bf2 = load_mesh_file(str(TF / 'brain_mesh.gii'))

k5r_base = dict(
    vertices=bv2, faces=bf2, roi_coords_df=K5r_coords,
    connectivity_matrix=str(K5r / 'connectivity_matrix.csv'),
    module_assignments=str(K5r / 'module_assignments.csv'),
    node_metrics=str(K5r / 'combined_metrics.csv'),
    node_size=K5_NODE, image_dpi=IMAGE_DPI, show_node_labels=False,
    node_roles=True, show_width_legend=False,
)
for key, vz in [('nodal_roles', 'nodes_only'), ('nodal_roles_edges', 'all')]:
    create_brain_connectivity_plot_with_modularity(
        **k5r_base, viz_type=vz, plot_title=f'Nodal roles ({vz})',
        multi_view=K5_ROLE_VIEWS, multi_view_panel_size=K5_ROLE_PANEL,
        save_path=str(HTML_TMP / f'{key}.html'),
        export_image=str(K5r_IMG / f'{key}_multiview.png'))
IPyImage(filename=str(K5r_IMG / 'nodal_roles_edges_multiview.png'))